In [3]:
df = spark.read.format("csv") \
.option("header","true") \
.option("inferSchema","true") \
.load("Files/Swiggy_Food_Delivery_Dataset.csv")

display(df)

StatementMeta(, 3547c044-a649-4057-ae9b-3b89f5cef731, 5, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 440f98d8-a703-44be-aad1-35d855f3aa41)

In [4]:
df.printSchema()

StatementMeta(, 3547c044-a649-4057-ae9b-3b89f5cef731, 6, Finished, Available, Finished, False)

root
 |-- order_id: string (nullable = true)
 |-- order_time: timestamp (nullable = true)
 |-- delivery_time: timestamp (nullable = true)
 |-- restaurant: string (nullable = true)
 |-- cuisine: string (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- rating: string (nullable = true)
 |-- location: string (nullable = true)
 |-- city: string (nullable = true)
 |-- ordered_qty: integer (nullable = true)
 |-- book_table: string (nullable = true)
 |-- online_order: string (nullable = true)
 |-- distance_km: double (nullable = true)



Checking the data first

In [5]:
from pyspark.sql.functions import *

StatementMeta(, 3547c044-a649-4057-ae9b-3b89f5cef731, 7, Finished, Available, Finished, False)

In [6]:
df.select(
    sum(col("rating").isNull().cast("int")).alias("rating_nulls"),
    sum(col("online_order").isNull().cast("int")).alias("online_order_nulls"),
    sum(col("book_table").isNull().cast("int")).alias("book_table_nulls")
).show()

StatementMeta(, 3547c044-a649-4057-ae9b-3b89f5cef731, 8, Finished, Available, Finished, False)

+------------+------------------+----------------+
|rating_nulls|online_order_nulls|book_table_nulls|
+------------+------------------+----------------+
|         811|              2069|            1963|
+------------+------------------+----------------+



In [7]:
from pyspark.sql.functions import *

StatementMeta(, 3547c044-a649-4057-ae9b-3b89f5cef731, 9, Finished, Available, Finished, False)

In [8]:
df = df.fillna({
    "online_order": "Unknown",
    "book_table": "Unknown"
})
display(df)

StatementMeta(, 3547c044-a649-4057-ae9b-3b89f5cef731, 10, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, cf3622a2-858b-46c6-b278-283e101cf3fb)

Cleaning the unwanted spaces in the column

In [9]:
df = df.withColumn(
    "city",
    trim(col("city"))
)

StatementMeta(, 3547c044-a649-4057-ae9b-3b89f5cef731, 11, Finished, Available, Finished, False)

In [10]:
df = df.withColumn(
    "city",
    initcap(col("city"))
)
display(df)

StatementMeta(, 3547c044-a649-4057-ae9b-3b89f5cef731, 12, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 14fcfbfe-e2ea-44e5-a6de-4602e848bf68)

fixing spelling mistakes

In [11]:
df = df.withColumn(
    "city",
    regexp_replace(col("city"), "Bnegaluru", "Bengaluru")
)

display(df)

StatementMeta(, 3547c044-a649-4057-ae9b-3b89f5cef731, 13, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, dbf5accc-63c1-4f61-899a-964600eee6c0)

verifying the result

In [12]:
df.select("city").distinct().show()

StatementMeta(, 3547c044-a649-4057-ae9b-3b89f5cef731, 14, Finished, Available, Finished, False)

+---------+
|     city|
+---------+
|Bengaluru|
+---------+



In [13]:
df = df.withColumn(
    "rating_numeric",
    split(col("rating"), "/")[0].cast("float")
)

display(df)

StatementMeta(, 3547c044-a649-4057-ae9b-3b89f5cef731, 15, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, da554350-f569-4b10-ba40-fa2271f02171)

In [14]:
df= df.fillna({"rating_numeric": 0})
display(df)

StatementMeta(, 3547c044-a649-4057-ae9b-3b89f5cef731, 16, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 7465d73b-fefd-4f9e-89e4-a8475d0f2ca7)

In [15]:
df = df.filter(col("total_amount")>0)

display(df)

StatementMeta(, 3547c044-a649-4057-ae9b-3b89f5cef731, 17, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 7a6efaaf-3c4e-48a2-a9e3-2899649208be)

In [16]:
df = df.withColumn(
    "delivery_time",
    when(col("online_order") == "No", None)
    .otherwise(col("delivery_time"))
)
display(df)

StatementMeta(, 3547c044-a649-4057-ae9b-3b89f5cef731, 18, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 7f2dc6ae-41db-477e-8dc3-c1ef2a76b3b8)

In [17]:
df.write.format("delta").mode("overwrite").saveAsTable("Cleaned_swiggy_orders")

StatementMeta(, 3547c044-a649-4057-ae9b-3b89f5cef731, 19, Finished, Available, Finished, False)